# Multivariate TimeGAN Analysis (WTI target focus)

This notebook trains TimeGAN on multivariate monthly data from `final_data.csv`, with `wti_ret` treated as the target of interest for evaluation/reporting.

It supports:
- Period selection (train/eval date ranges)
- Basic hyperparameter tuning
- Fidelity evaluation (marginal, temporal, joint, discriminative)
- Synthetic sequence generation and export

In [ ]:
# Optional install (uncomment if needed)
# %pip install -q ydata-synthetic scikit-learn scipy statsmodels seaborn

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
os.environ.pop('TF_USE_LEGACY_KERAS', None)

from pathlib import Path
import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import ks_2samp, wasserstein_distance
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from statsmodels.tsa.stattools import acf

from ydata_synthetic.synthesizers import ModelParameters, TrainParameters
from ydata_synthetic.synthesizers.timeseries import TimeSeriesSynthesizer

In [ ]:
# =========================
# CONFIG
# =========================
CFG = {
    # Data
    'data_path': '../data/final_data.csv',
    'date_col': 'Date',
    'y_col': 'wti_ret',

    # Period selection (monthly)
    'train_start': '1990-01-01',
    'train_end': '2018-12-01',
    'eval_start': '2019-01-01',
    'eval_end': '2026-01-01',

    # Sequence/sampling
    'sequence_length': 24,
    'n_synth_eval': None,   # None = match real eval window count
    'n_synth_export': 50,

    # TimeGAN base hyperparameters
    'noise_dim': 32,
    'layers_dim': 128,
    'batch_size': 64,
    'learning_rate': 5e-4,
    'epochs': 1200,

    # Apple Silicon speed knobs
    'use_legacy_adam': True,
    'patch_fast_train_loop': True,
    'embedding_steps_ratio': 0.6,
    'supervisor_steps_ratio': 0.6,
    'joint_steps_ratio': 1.0,

    # Tuning search space (small starter grid)
    'tune_grid': {
        'sequence_length': [12, 24, 36],
        'noise_dim': [16, 32],
        'layers_dim': [64, 128],
        'batch_size': [32, 64],
        'learning_rate': [1e-3, 5e-4],
        'epochs': [800]
    },
    'max_tune_runs': 6,
    'random_seed': 42,

    # Output
    'output_dir': '../data/timegan_outputs'
}

np.random.seed(CFG['random_seed'])
Path(CFG['output_dir']).mkdir(parents=True, exist_ok=True)
CFG

{'data_path': '../data/final_data.csv',
 'date_col': 'Date',
 'y_col': 'wti_ret',
 'train_start': '1990-01-01',
 'train_end': '2018-12-01',
 'eval_start': '2019-01-01',
 'eval_end': '2026-01-01',
 'sequence_length': 24,
 'n_synth_eval': None,
 'n_synth_export': 50,
 'noise_dim': 32,
 'layers_dim': 128,
 'batch_size': 64,
 'learning_rate': 0.0005,
 'epochs': 1500,
 'tune_grid': {'sequence_length': [12, 24, 36],
  'noise_dim': [16, 32],
  'layers_dim': [64, 128],
  'batch_size': [32, 64],
  'learning_rate': [0.001, 0.0005],
  'epochs': [800]},
 'max_tune_runs': 8,
 'random_seed': 42,
 'output_dir': '../data/timegan_outputs'}

In [ ]:
# =========================
# Helpers
# =========================
def apply_timegan_speed_patches(cfg):
    from keras.optimizers import legacy as legacy_optimizers
    from ydata_synthetic.synthesizers.timeseries.timegan import model as timegan_model

    if cfg.get('use_legacy_adam', True):
        timegan_model.Adam = legacy_optimizers.Adam

    if cfg.get('patch_fast_train_loop', True):
        TimeGAN = timegan_model.TimeGAN
        if not getattr(TimeGAN, '_fast_patch_applied', False):
            def train_fast(self, data, train_steps):
                self.define_gan()

                train_steps = int(train_steps)
                e_steps = max(1, int(train_steps * cfg.get('embedding_steps_ratio', 1.0)))
                s_steps = max(1, int(train_steps * cfg.get('supervisor_steps_ratio', 1.0)))
                j_steps = max(1, int(train_steps * cfg.get('joint_steps_ratio', 1.0)))

                data_iter = self.get_batch_data(data, n_windows=len(data))
                noise_iter = self.get_batch_noise()

                autoencoder_opt = timegan_model.Adam(learning_rate=self.g_lr)
                for _ in timegan_model.tqdm(range(e_steps), desc='Embedding network training'):
                    X_ = next(data_iter)
                    _ = self.train_autoencoder(X_, autoencoder_opt)

                supervisor_opt = timegan_model.Adam(learning_rate=self.g_lr)
                for _ in timegan_model.tqdm(range(s_steps), desc='Supervised network training'):
                    X_ = next(data_iter)
                    _ = self.train_supervisor(X_, supervisor_opt)

                generator_opt = timegan_model.Adam(learning_rate=self.g_lr)
                embedder_opt = timegan_model.Adam(learning_rate=self.g_lr)
                discriminator_opt = timegan_model.Adam(learning_rate=self.d_lr)

                for _ in timegan_model.tqdm(range(j_steps), desc='Joint networks training'):
                    for _ in range(2):
                        X_ = next(data_iter)
                        Z_ = next(noise_iter)
                        _ = self.train_generator(X_, Z_, generator_opt)
                        _ = self.train_embedder(X_, embedder_opt)

                    X_ = next(data_iter)
                    Z_ = next(noise_iter)
                    step_d_loss = self.discriminator_loss(X_, Z_)
                    if step_d_loss > 0.15:
                        _ = self.train_discriminator(X_, Z_, discriminator_opt)

            TimeGAN.train = train_fast
            TimeGAN._fast_patch_applied = True


def load_and_validate_data(cfg):
    df = pd.read_csv(cfg['data_path'])
    df[cfg['date_col']] = pd.to_datetime(df[cfg['date_col']])
    df = df.sort_values(cfg['date_col']).reset_index(drop=True)

    feature_cols = [c for c in df.columns if c != cfg['date_col']]
    assert cfg['y_col'] in feature_cols, f"{cfg['y_col']} not found in features"

    idx = pd.date_range(df[cfg['date_col']].min(), df[cfg['date_col']].max(), freq='MS')
    missing_months = idx.difference(df[cfg['date_col']])

    profile = {
        'shape': df.shape,
        'date_min': str(df[cfg['date_col']].min().date()),
        'date_max': str(df[cfg['date_col']].max().date()),
        'n_missing_values': int(df[feature_cols].isna().sum().sum()),
        'duplicate_dates': int(df[cfg['date_col']].duplicated().sum()),
        'missing_month_count': int(len(missing_months)),
        'missing_month_sample': [str(x.date()) for x in missing_months[:5]],
        'feature_cols': feature_cols
    }

    return df, feature_cols, profile


def select_period(df, date_col, start, end):
    m = (df[date_col] >= pd.Timestamp(start)) & (df[date_col] <= pd.Timestamp(end))
    return df.loc[m].copy()


def make_windows(array_2d, seq_len):
    n = array_2d.shape[0]
    if n < seq_len:
        return np.empty((0, seq_len, array_2d.shape[1]))
    return np.stack([array_2d[i:i+seq_len] for i in range(n - seq_len + 1)], axis=0)


def flatten_windows(w):
    return w.reshape(w.shape[0], -1)


def train_timegan(train_df, feature_cols, params):
    apply_timegan_speed_patches(CFG)

    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(train_df[feature_cols].values)

    # In ydata-synthetic, number_sequences sets the model's input feature dimension
    # (despite the misleading name). Must equal the number of feature columns.
    model_args = ModelParameters(
        batch_size=params['batch_size'],
        lr=params['learning_rate'],
        noise_dim=params['noise_dim'],
        layers_dim=params['layers_dim']
    )
    train_args = TrainParameters(
        epochs=params['epochs'],
        sequence_length=params['sequence_length'],
        number_sequences=len(feature_cols)
    )

    synth = TimeSeriesSynthesizer(modelname='timegan', model_parameters=model_args)
    synth.fit(pd.DataFrame(train_scaled, columns=feature_cols), train_args, num_cols=feature_cols)

    return synth, scaler


def sample_synthetic(synth, scaler, n_samples, seq_len, n_features):
    s = np.asarray(synth.sample(n_samples))
    if s.ndim != 3:
        raise ValueError(f'Unexpected synthetic shape: {s.shape}')
    s_inv = scaler.inverse_transform(s.reshape(-1, n_features)).reshape(-1, seq_len, n_features)
    return s_inv


def eval_metrics(real_w, synth_w, feature_cols, y_col, real_series_2d=None, max_acf_lag=12):
    """
    real_series_2d: original contiguous eval series (T x n_features).
                    Used for ACF to avoid artificial autocorrelation from
                    concatenated overlapping windows.
    """
    real_flat = real_w.reshape(-1, real_w.shape[-1])
    synth_flat = synth_w.reshape(-1, synth_w.shape[-1])

    # Marginal distances
    marginal = []
    for j, c in enumerate(feature_cols):
        r = real_flat[:, j]
        s = synth_flat[:, j]
        ks = ks_2samp(r, s).statistic
        wd = wasserstein_distance(r, s)
        marginal.append({'feature': c, 'ks': float(ks), 'wasserstein': float(wd)})
    marginal_df = pd.DataFrame(marginal).sort_values('feature').reset_index(drop=True)

    # Joint (correlation matrix Frobenius distance)
    corr_real = pd.DataFrame(real_flat, columns=feature_cols).corr().values
    corr_synth = pd.DataFrame(synth_flat, columns=feature_cols).corr().values
    corr_fro = float(np.linalg.norm(corr_real - corr_synth, ord='fro'))

    # Temporal fidelity: ACF on the original contiguous eval series
    y_idx = feature_cols.index(y_col)
    real_y = real_series_2d[:, y_idx] if real_series_2d is not None else real_flat[:, y_idx]
    synth_y = synth_flat[:, y_idx]
    acf_r = acf(real_y, nlags=max_acf_lag, fft=True)
    acf_s = acf(synth_y, nlags=max_acf_lag, fft=True)
    acf_l2 = float(np.linalg.norm(acf_r - acf_s))

    # Discriminative score
    xr = flatten_windows(real_w)
    xs = flatten_windows(synth_w)
    X = np.vstack([xr, xs])
    y = np.hstack([np.zeros(xr.shape[0]), np.ones(xs.shape[0])])
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_tr, y_tr)
    auc = float(roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1]))

    summary = {
        'avg_ks': float(marginal_df['ks'].mean()),
        'avg_wasserstein': float(marginal_df['wasserstein'].mean()),
        'corr_fro': corr_fro,
        'acf_l2_y': acf_l2,
        'disc_auc': auc,
        'disc_gap_abs_auc_minus_0_5': float(abs(auc - 0.5)),
    }

    return summary, marginal_df, corr_real, corr_synth, acf_r, acf_s

In [8]:
# =========================
# Load + period setup
# =========================
df, feature_cols, profile = load_and_validate_data(CFG)
display(pd.DataFrame([profile]))

train_df = select_period(df, CFG['date_col'], CFG['train_start'], CFG['train_end'])
eval_df = select_period(df, CFG['date_col'], CFG['eval_start'], CFG['eval_end'])

print('Train rows:', len(train_df), ' | Eval rows:', len(eval_df))
print('Features:', feature_cols)

assert len(train_df) >= CFG['sequence_length'], 'Train period too short for sequence length'
assert len(eval_df) >= CFG['sequence_length'], 'Eval period too short for sequence length'

,shape,date_min,date_max,n_missing_values,duplicate_dates,missing_month_count,missing_month_sample,feature_cols
0,"(480, 10)",1986-02-01,2026-01-01,0,0,0,[],"[AUD_USD_ret, CAD_USD_ret, NZD_USD_ret, ZAR_US..."


Train rows: 348  | Eval rows: 85
Features: ['AUD_USD_ret', 'CAD_USD_ret', 'NZD_USD_ret', 'ZAR_USD_ret', 'CPI', 'TB3MS', 'M1', 'M2', 'wti_ret']


In [ ]:
# =========================
# Train TimeGAN (single run)
# =========================
params = {
    'sequence_length': CFG['sequence_length'],
    'noise_dim':       CFG['noise_dim'],
    'layers_dim':      CFG['layers_dim'],
    'batch_size':      CFG['batch_size'],
    'learning_rate':   CFG['learning_rate'],
    'epochs':          CFG['epochs'],
}

n_train_windows = len(train_df) - params['sequence_length'] + 1
print(f"Params: {params}")
print(f"Training windows available: {n_train_windows}")

synth_model, scaler = train_timegan(train_df, feature_cols, params)

# =========================
# Evaluate on eval set
# =========================
# Build real eval windows directly from raw values — no redundant scale/unscale.
real_eval_w = make_windows(eval_df[feature_cols].values, params['sequence_length'])
print(f"\nReal eval windows: {real_eval_w.shape}")  # (n_windows, seq_len, n_features)

n_synth_eval = CFG['n_synth_eval'] or len(real_eval_w)
synth_eval_w = sample_synthetic(
    synth_model, scaler,
    n_samples=n_synth_eval,
    seq_len=params['sequence_length'],
    n_features=len(feature_cols)
)
print(f"Synthetic eval windows: {synth_eval_w.shape}")

# Pass the original contiguous eval series for a correct ACF computation.
real_eval_series = eval_df[feature_cols].values

summary, marginal_df, corr_real, corr_synth, acf_r, acf_s = eval_metrics(
    real_eval_w, synth_eval_w, feature_cols, CFG['y_col'],
    real_series_2d=real_eval_series
)

print("\nEvaluation metrics:")
display(pd.DataFrame([summary]))

In [ ]:
# =========================
# Diagnostics
# =========================
display(marginal_df.sort_values('ks'))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(corr_real, ax=axes[0], cmap='coolwarm', center=0,
            xticklabels=feature_cols, yticklabels=feature_cols)
axes[0].set_title('Real correlation matrix (eval)')

sns.heatmap(corr_synth, ax=axes[1], cmap='coolwarm', center=0,
            xticklabels=feature_cols, yticklabels=feature_cols)
axes[1].set_title('Synthetic correlation matrix')
plt.tight_layout()
plt.show()

lags = np.arange(len(acf_r))
plt.figure(figsize=(8, 4))
plt.plot(lags, acf_r, marker='o', label='Real ACF (wti_ret)')
plt.plot(lags, acf_s, marker='x', label='Synth ACF (wti_ret)')
plt.axhline(0, color='grey', linewidth=1)
plt.title('ACF comparison on wti_ret (contiguous eval series vs synthetic)')
plt.xlabel('Lag')
plt.ylabel('ACF')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Generate and export final synthetic sequences
# =========================
final_synth_w = sample_synthetic(
    synth_model, scaler,
    n_samples=CFG['n_synth_export'],
    seq_len=params['sequence_length'],
    n_features=len(feature_cols)
)
print(f"Synthetic output shape (samples, seq_len, n_features): {final_synth_w.shape}")
# Each sample is one independent sequence of length seq_len (={params['sequence_length']} months).

records = []
for i in range(final_synth_w.shape[0]):
    for t in range(final_synth_w.shape[1]):
        rec = {'sample_id': i + 1, 'step': t + 1}
        for j, c in enumerate(feature_cols):
            rec[c] = float(final_synth_w[i, t, j])
        records.append(rec)

synth_long_df = pd.DataFrame(records)
out_synth = Path(CFG['output_dir']) / 'timegan_synthetic_sequences_long.csv'
synth_long_df.to_csv(out_synth, index=False)

best_cfg_path = Path(CFG['output_dir']) / 'timegan_best_run_summary.json'
with open(best_cfg_path, 'w') as f:
    json.dump({'config': CFG, 'params': params, 'eval_summary': summary}, f, indent=2)

print('Saved synthetic sequences:', out_synth)
print('Saved run summary:', best_cfg_path)
display(synth_long_df.head())

## Notes on interpretation
- Lower `avg_ks`, `avg_wasserstein`, `corr_fro`, `acf_l2_y`, and `disc_gap_abs_auc_minus_0_5` are better.
- `disc_auc` near 0.5 means synthetic windows are hard to distinguish from real windows.
- This is joint generative modeling; `wti_ret` is highlighted for diagnostics, not conditionally generated.